# Warp Drive Sky-Flight (Colab GPU)

Render high-quality videos of what the night sky looks like through
the front window of a spacecraft as it accelerates from rest into
superluminal velocities inside an Alcubierre / Natário warp bubble.

**Pipeline:**

1. Clone the WarpDrives repo and install in editable mode
2. Switch JAX to the GPU backend (Colab T4/A100 friendly)
3. Download an equirectangular Milky Way panorama (ESO public-domain) — or use the procedural starfield
4. For each velocity in a smooth ramp 0 → v_max, fire backward null geodesics through the warp metric, vectorised across all pixels
5. Save individual frames + assemble an MP4 / GIF

Suggested runtime: **GPU (T4)** — set via `Runtime → Change runtime type` before running.

## 1. Install dependencies

Skip the JAX-CUDA wheels if you're not on a GPU runtime.

In [ ]:
!pip install -q --upgrade "numpy>=2.0" "scipy>=1.13"
!pip install -q -U jax jaxlib
!pip install -q imageio imageio-ffmpeg pyyaml click tqdm pyvista==0.43.10

import os, sys
if not os.path.exists('WarpDrives'):
    !git clone https://github.com/mthiel74/WarpDrives.git
%cd WarpDrives
!pip install -q -e .
sys.path.insert(0, os.getcwd())

In [ ]:
# Confirm JAX sees the GPU.
import jax
print('JAX:', jax.__version__)
print('Devices:', jax.devices())
print('Default backend:', jax.default_backend())

## 2. Sky background

Two options.  Either:

- **Procedural starfield** — no download, fast, deterministic.
- **Real Milky Way panorama** — drop in any equirectangular 2:1 sky image.  ESO's *GigaGalaxy* panorama is freely available; the cell below downloads a moderate-resolution version.

In [ ]:
# Option A: download an ESO Milky Way panorama (CC BY 4.0 / Brunier).
# Comment out if you'd rather use the procedural sky in option B.
import urllib.request, os

MW_URL = 'https://cdn.eso.org/images/large/eso0932a.jpg'
MW_PATH = 'images/milky_way_panorama.jpg'
os.makedirs('images', exist_ok=True)
if not os.path.exists(MW_PATH):
    print('Downloading Milky Way panorama (ESO/Brunier)...')
    urllib.request.urlretrieve(MW_URL, MW_PATH)
print('Sky image at:', MW_PATH, 'size on disk:', os.path.getsize(MW_PATH) // 1024, 'KB')

In [ ]:
# Option B: procedural starfield (works without any download).
import numpy as np
from warpbubblesim.viz.skybackground import (
    make_image_sky, make_procedural_starfield,
)

USE_REAL_PANORAMA = os.path.exists(MW_PATH)
if USE_REAL_PANORAMA:
    sky = make_image_sky(MW_PATH, rotation_deg=0.0, gain=1.6)
    print('Using ESO Milky Way panorama as background')
else:
    fov_deg = 90.0
    res = 512
    sky = make_procedural_starfield(
        n_stars=15000, seed=2026,
        star_radius_px=0.6, fov_scale=np.deg2rad(fov_deg) / res,
    )
    print('Using procedural starfield')

## 3. JAX renderer — quick sanity render

First a single small frame to JIT-compile the integrator (compile-once, run-many).

In [ ]:
from warpbubblesim.viz.skyrender_jax import (
    JaxRenderConfig, render_frame_jax,
)
import time, matplotlib.pyplot as plt

small_cfg = JaxRenderConfig(width=128, height=128, fov_deg=90.0,
                            n_steps=200, dlam=0.15, chunk_size=4096)
params0 = dict(v0=0.95, R=1.0, sigma=8.0, shape='tanh')
t0 = time.time()
preview = render_frame_jax('alcubierre', params0, sky, small_cfg)
print(f'First frame (incl. JIT compile): {time.time()-t0:.1f}s')

t0 = time.time()
preview2 = render_frame_jax('alcubierre', dict(params0, v0=2.5), sky, small_cfg)
print(f'Second frame (cached JIT): {time.time()-t0:.2f}s')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.clip(preview, 0, 1)); axes[0].set_title('v=0.95')
axes[1].imshow(np.clip(preview2, 0, 1)); axes[1].set_title('v=2.5')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## 4. Full HQ velocity-sweep animation

Render at high resolution across a smooth velocity ramp.  On a T4 GPU each
256×256 frame should be a few seconds; 512×512 a bit more.  Drop the
resolution if memory is tight.

In [ ]:
RESOLUTION = 384
FOV_DEG = 90.0
N_FRAMES = 60
V_MAX = 4.0
METRIC = 'alcubierre'   # or 'natario'

# Smooth ramp: ease-in-out cosine — feels like an actual throttle
tt = np.linspace(0, 1, N_FRAMES)
ease = 0.5 - 0.5 * np.cos(np.pi * tt)
velocities = V_MAX * ease

cfg = JaxRenderConfig(
    width=RESOLUTION, height=RESOLUTION, fov_deg=FOV_DEG,
    n_steps=240, dlam=0.15, chunk_size=8192,
)
if USE_REAL_PANORAMA:
    sky = make_image_sky(MW_PATH, rotation_deg=0.0, gain=1.6)
else:
    sky = make_procedural_starfield(
        n_stars=18000, seed=2026,
        star_radius_px=0.55, fov_scale=np.deg2rad(FOV_DEG) / RESOLUTION,
    )

frames = []
for i, v in enumerate(velocities):
    t0 = time.time()
    img = render_frame_jax(METRIC, dict(v0=float(v), R=1.0, sigma=8.0, shape='tanh'),
                           sky, cfg)
    frames.append(img)
    print(f'frame {i+1}/{N_FRAMES} v={v:.3f} {time.time()-t0:.1f}s')

In [ ]:
from warpbubblesim.viz.skyrender import save_frames_as_animation
import os
os.makedirs('out', exist_ok=True)

# Smooth tail-and-back: pause briefly at high v then ramp back to 0
extended = list(frames) + list(frames[-1:]) * 6 + list(reversed(frames))

save_frames_as_animation(extended, 'out/warp_skyflight.mp4', fps=24)
save_frames_as_animation(extended, 'out/warp_skyflight.gif', fps=12)
print('Saved out/warp_skyflight.mp4 and .gif')

# Display inline
from IPython.display import Video
Video('out/warp_skyflight.mp4', embed=True, width=512)

## 5. Side-by-side: Alcubierre vs Natário

Two metrics, one velocity ramp, both rendered with the same sky and camera.

In [ ]:
comp_cfg = JaxRenderConfig(width=256, height=256, fov_deg=90.0,
                           n_steps=240, dlam=0.15, chunk_size=8192)
comp_velocities = V_MAX * (0.5 - 0.5 * np.cos(np.pi * np.linspace(0, 1, 30)))

comp_frames = []
for i, v in enumerate(comp_velocities):
    t0 = time.time()
    a = render_frame_jax('alcubierre', dict(v0=float(v), R=1.0, sigma=8.0, shape='tanh'), sky, comp_cfg)
    n = render_frame_jax('natario',    dict(v0=float(v), R=1.0, sigma=8.0),                  sky, comp_cfg)
    side = np.concatenate([a, n], axis=1)
    comp_frames.append(side)
    print(f'compare {i+1}/{len(comp_velocities)} v={v:.3f} {time.time()-t0:.1f}s')

save_frames_as_animation(comp_frames, 'out/warp_skyflight_comparison.mp4', fps=18)
Video('out/warp_skyflight_comparison.mp4', embed=True, width=600)

## Notes & tips

- The renderer integrates a **fixed-step RK4**.  ``n_steps × dlam`` should
  exceed the time the slowest ray needs to leave the bubble influence;
  any extra steps are spent in flat space and don't change the
  asymptotic direction.
- ``chunk_size`` controls how many rays are vmapped together.  On a T4,
  4–8k is comfortable; on an A100 you can push it to 16–32k for a
  noticeable speedup.
- Memory blows up linearly in ``n_stars × pixels`` for the procedural
  sky; the equirectangular ``make_image_sky`` is constant-memory.
- The Alcubierre interior observer is *naturally co-moving* — there's
  no SR boost between observer and bubble, so aberration in the
  conventional sense is mild.  The dramatic deformations show up at
  oblique angles where rays pass through more of the bubble wall.